# JsonFileRepository Performance Benchmark & Analysis

This notebook benchmarks the performance of the `JsonFileRepository` used in `multiagent-mcp` and analyzes it using a theoretical performance model.

## Mathematical Model

The latency of file-based storage operations where all data is stored in a single JSON file can be modeled as:

$$ L_{total} = T_{read} + T_{parse} + T_{modify} + T_{serialize} + T_{write} $
Since the entire dataset is read and written for every operation:

$$ T_{op}(N) = O(N) $
Where $N$ is the number of records. This linear scaling is the primary bottleneck.

In [ ]:
import shutil
import sys
import tempfile
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# Ensure src is in path
sys.path.append("../src")

from pydantic import BaseModel

from mcp_core.json_repository import JsonFileRepository

In [ ]:
# Setup Temporary Directory for Benchmarks
temp_dir = tempfile.mkdtemp()
db_path = Path(temp_dir) / "benchmark.json"

print(f"Benchmarking DB: {db_path}")


# Define a simple model for testing
class BenchmarkModel(BaseModel):
    id: str
    data: str
    version: int = 0


# Initialize Repo
repo = JsonFileRepository(BenchmarkModel, str(db_path))

In [ ]:
# Benchmark Write Performance
record_counts = [10, 50, 100, 200]
# Note: Higher counts (e.g., 1000) become very slow with O(N) file rewrite on every save.

write_times = []
read_times = []

for count in record_counts:
    # Cleanup and Init
    if db_path.exists():
        db_path.unlink()

    repo = JsonFileRepository(BenchmarkModel, str(db_path))

    # WRITE TEST (Sequential Inserts)
    # We measure the time to insert the LAST item, as that represents the cost at size N
    # To average it, we can insert a batch.

    start_time = time.time()
    for i in range(count):
        item = BenchmarkModel(id=str(i), data=f"content_{i}" * 10)
        repo.save(item)

    total_duration = time.time() - start_time
    avg_write_time = total_duration / count * 1000  # This averages 0..N growth
    write_times.append(avg_write_time)

    # READ TEST (Random Access)
    start_time = time.time()
    for i in range(count):
        _ = repo.get(str(i))

    duration = time.time() - start_time
    read_times.append(duration / count * 1000)

    print(f"N={count}: Avg Write={avg_write_time:.2f}ms, Avg Read={read_times[-1]:.2f}ms")

## Sensitivity Analysis & Projections

Based on the collected data, we can project the latency for larger datasets ($N=1k, 10k, 100k$).

### Projection Model
We assume $T(N) = k \cdot N + c$.

In [ ]:
# Linear Regression for Projection
coeffs = np.polyfit(record_counts, write_times, 1)
poly = np.poly1d(coeffs)

projected_counts = [1000, 10000, 100000]
projected_times = poly(projected_counts)

print("Projected Write Latencies:")
for n, t in zip(projected_counts, projected_times, strict=False):
    print(f"N={n}: {t:.2f} ms")

plt.figure(figsize=(10, 6))
plt.plot(record_counts, write_times, "bo-", label="Measured Write")
plt.plot(projected_counts, projected_times, "r--", label="Projected Write")
plt.xscale("log")
plt.yscale("log")
plt.title("Sensitivity Analysis: Latency vs Scale (Log-Log)")
plt.xlabel("Number of Records (N)")
plt.ylabel("Latency (ms)")
plt.grid(True)
plt.legend()
plt.show()

### References
1. **Big O Notation**: Knuth, D. E. (1976). *The Art of Computer Programming, Volume 3: Sorting and Searching*.
2. **JSON Performance**: *Parsing JSON is a Minefield*, Seriot, N. (2018). http://seriot.ch/projects/parsing_json.html
3. **File I/O Latency**: Arpaci-Dusseau, R. H., & Arpaci-Dusseau, A. C. (2018). *Operating Systems: Three Easy Pieces*. 

In [ ]:
# Cleanup
shutil.rmtree(temp_dir)
print("Cleanup complete.")